In [1]:
import numpy as np
import torch
import cv2
import json
import matplotlib.pyplot as plt
from pathlib import Path
from project_root import DATASETS_ROOT

from typing import Any

In [2]:
coco_root = DATASETS_ROOT / "elephants/segmentation/coco"

# ds_type = "train"
ds_type = "val"

annotations_dir = coco_root / f"annotations"
image_dir = coco_root / f"{ds_type}2017"
segmentation_dir = coco_root / f"panoptic_{ds_type}2017"

data_file = annotations_dir / f"panoptic_{ds_type}2017.json"
with data_file.open() as f:
    data = json.load(f)

In [3]:
print(f"Images: {len(data['images'])}")
print(f"Annotations: {len(data['annotations'])}")
print(f"Categories: {len(data['categories'])}")

Images: 5000
Annotations: 5000
Categories: 133


In [4]:
# Ditch extra categories
elephant_category = [c for c in data["categories"] if c["name"] == "elephant"][0]
original_category_id = elephant_category["id"]
elephant_category["id"] = 1

background_category = {
    "supercategory": "background",
    "isthing": 0,
    "id": 0,
    "name": "background",
}
data["categories"] = [background_category, elephant_category]
print(data["categories"])

[{'supercategory': 'background', 'isthing': 0, 'id': 0, 'name': 'background'}, {'supercategory': 'animal', 'isthing': 1, 'id': 1, 'name': 'elephant'}]


In [5]:
def has_elephants(annotation):
    for segment in annotation["segments_info"]:
        if segment["category_id"] == original_category_id:
            return True
    return False


def force_segment_info(segment):
    if segment["category_id"] == original_category_id:
        segment["category_id"] = 1
    else:
        segment["category_id"] = 0
    return segment


def force_annotation(a):
    a["segments_info"] = [force_segment_info(segment) for segment in a["segments_info"]]
    return a


data["annotations"] = [
    force_annotation(a) for a in data["annotations"] if has_elephants(a)
]
print(f"Annotations: {len(data['annotations'])}")

Annotations: 89


In [6]:
# Make sure that all annotations have a background mapping
for annotation in data["annotations"]:
    has_bg = False
    for segment in annotation["segments_info"]:
        if segment["id"] == 0:
            has_bg = True
            break
    if not has_bg:
        annotation["segments_info"].append(
            {
                "id": 0,
                "category_id": 0,
                "iscrowd": 1,
                "bbox": [0, 0, 1, 1],
                "area": 1,
            },
        )

In [7]:
valid_image_ids = {a["image_id"] for a in data["annotations"]}
data["images"] = [i for i in data["images"] if i["id"] in valid_image_ids]
print(f"Images: {len(data['images'])}")

Images: 89


In [8]:
# Update image path
for image in data["images"]:
    image["file_name"] = str(
        Path("..") / (image_dir / image["file_name"]).relative_to(coco_root)
    )

# Update segmentation path
for annotation in data["annotations"]:
    annotation["file_name"] = str(
        Path("..") / (segmentation_dir / annotation["file_name"]).relative_to(coco_root)
    )

In [9]:
output_file = (
    coco_root
    / "annotations"
    / data_file.name.replace("panoptic_", "panoptic_elephants_")
)
with output_file.open("w") as f:
    json.dump(data, f, indent=1)
print(
    f"Written to {str(output_file)} ({output_file.lstat().st_size / 1024/1024:.2f} mb)"
)

Written to /home/dherrera/data/elephants/segmentation/coco/annotations/panoptic_elephants_val2017.json (0.16 mb)
